# Runde 1b · L-CAL auf CPU mit hohem RAM · H_tq-Variante

In Konto A, in einer eigenen CPU-Laufzeit mit hohem RAM öffnen. Erst starten,
nachdem HTQ_START die beiden Variantenordner als bereit gemeldet hat; danach
kann L-CAL parallel zum GPU-Lauf rechnen. Der eingefrorene L-CAL-Code und die
synthetischen Kalibrierungsdaten bleiben unverändert. Für den aktuellen
Generator ist keine belastbare Laufzeit gemessen; mehrere Stunden einplanen.

Ergebnis: `dsse_runs/variants/htq_v2/r1b_lcal/l_cal.json`. Diese Datei vor ANA
dem Koordinator zur Übernahme melden. Zwischenstände bleiben im Varianten-Cache.


In [ ]:
import os, sys, subprocess, json, glob, time, hashlib, shutil
from google.colab import drive; drive.mount("/content/drive")
REPO = "/content/DSSE_htq_lcal"; FREEZE = "4de7a3b41169704dd811e11965727e0e64d6b8f9"
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "codex/r1b-htq-continuation", "https://github.com/Erikiss/DSSE", REPO], check=True)
R = f"{REPO}/analysis/ju_space/round1b"
d = subprocess.run(["git", "-C", REPO, "diff", "--stat", FREEZE, "HEAD", "--", f"{R}/r1b_l_cal.py", f"{R}/r1b_l_stats.py", f"{R}/r1b_l_rules.py"],
                   capture_output=True, text=True, check=True).stdout
assert not d.strip(), "L-CAL-Code weicht vom Freeze-Commit ab: " + d
print("Code = Freeze-Stand; Kerne:", os.cpu_count())
RUNS = "/content/drive/MyDrive/dsse_runs/variants/htq_v2"
clean_env = {k: v for k, v in os.environ.items() if k not in ("R1B_H_POLICY", "R1B_RUNS_B", "R1B_XFER_A", "R1B_SOURCE_RUNS", "R1B_CAPTURE_HIDDEN")}
R1B, TEST_ENV = R, clean_env
# Variantenwerkzeug: Ausgabe und Fehler mit Konto sichtbar streamen.
import textwrap
VARIANT_TOOL = f"{R1B}/r1b_htq_variant.py"
def run_variant(action, path, account):
    def show(message):
        for line in str(message).splitlines() or [""]:
            for wrapped in textwrap.wrap(line, width=72, replace_whitespace=False,
                                         drop_whitespace=False, break_on_hyphens=False) or [""]:
                print(wrapped, flush=True)
    show(f"[Konto {account}] {action}: {path}")
    command = [sys.executable, "-u", VARIANT_TOOL, action, path, account]
    try:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env=TEST_ENV)
    except OSError as error:
        show(f"[Konto {account}] {action}: Start fehlgeschlagen: {error}")
        raise RuntimeError(f"Konto {account}: {action} konnte nicht starten.") from error
    try:
        for line in process.stdout:
            show(line.rstrip("\r\n"))
        code = process.wait()
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        show(f"[Konto {account}] {action}: unterbrochen; Kindprozess beendet.")
        raise
    finally:
        process.stdout.close()
    show(f"[Konto {account}] {action}: Exit {code}")
    if code:
        raise RuntimeError(f"Konto {account}: {action} fehlgeschlagen (Exit {code}). Details oben.")
    return code

def check_variant(path, account):
    return run_variant("check", path, account)

check_variant(RUNS, "A")
pair = json.load(open(os.path.join(RUNS, "_htq_pair_ready.json")))
assert pair.get("variant") == "htq_v2" and pair.get("accounts") == ["A", "B"], "Zuerst HTQ_START vorbereiten."
OUT = f"{RUNS}/r1b_lcal"; CACHE = f"{OUT}/cache"; os.makedirs(CACHE, exist_ok=True)


In [ ]:
# Teilstücke parallel (je 2 Threads): (a) nach Profilen, dann die übrigen Punkte; jedes fertige Teilstück liegt sofort in Drive
PART = f'''
import os, sys, pickle
sys.path.insert(0, "{R}")
import r1b_l_cal as LC
prof, out = sys.argv[1], sys.argv[2]
os.environ["R1B_LCAL_A_PROFILES"] = prof
r = LC.item_a(False, 5); pickle.dump(r, open(out + ".tmp", "wb")); os.replace(out + ".tmp", out)
print("fertig", prof, r["hits"], flush=True)
'''
open("/content/a_part.py", "w").write(PART)
ncpu = os.cpu_count() or 2
jobs = []
for prof in ("localist", "redundant", "dense", "dthr", "xor3"):
    p = f"{CACHE}/a_{prof}.pkl"
    if not os.path.exists(p):
        jobs.append(["python3", "/content/a_part.py", prof, p])
WRAP = f"{REPO}/results/ju_space/plans/runde1b/arbeitsstand/lcal/lcal_resumable.py"
rest = "".join(k for k in "bcdefghijklmn" if not os.path.exists(f"{CACHE}/{k}.pkl"))
if rest:
    jobs.append(["python3", WRAP, f"{OUT}/l_cal.json", CACHE, "--items", rest])
thr = str(max(1, ncpu // max(1, min(len(jobs), ncpu))))
procs = []
for j in jobs:
    procs.append(subprocess.Popen(j, env=dict(clean_env, OMP_NUM_THREADS=thr), stdout=open(f"/content/log_{len(procs)}.txt", "w"), stderr=subprocess.STDOUT))
    while sum(p.poll() is None for p in procs) >= max(1, ncpu // int(thr)):
        time.sleep(30)
while any(p.poll() is None for p in procs):
    time.sleep(60); print(time.strftime("%H:%M"), "fertig in Drive:", sorted(os.path.basename(x) for x in glob.glob(f"{CACHE}/*.pkl")), flush=True)
print("Teilstücke fertig.")

In [ ]:
# (a) zusammenführen und l_cal.json schreiben (Torprüfung G_cal_L1..L5 durch r1b_l_cal.run)
import pickle
sys.path.insert(0, R); import r1b_l_cal as LC
parts = [pickle.load(open(f"{CACHE}/a_{p}.pkl", "rb")) for p in ("localist", "redundant", "dense", "dthr", "xor3")]
hits = {k: v for x in parts for k, v in x["hits"].items()}; need = 4
a = dict(hits=hits, seeds=5, trace=sum((x["trace"] for x in parts), []), pass_=all(hits[p] >= need for p in hits))
pickle.dump(a, open(f"{CACHE}/a.pkl", "wb"))
subprocess.run(["python3", WRAP, f"{OUT}/l_cal.json", CACHE], check=True)
print(json.dumps(json.load(open(f"{OUT}/l_cal.json"))["gates"], indent=1))